In [45]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt

# Load the data into a pandas DataFrame (assuming it's in CSV format)
data = pd.read_csv('merged_bid_ask_ohlcv_data.csv', parse_dates=['timestamp'], index_col='timestamp')

# We only need the OHLC (open, high, low, close) and bid_price_1, ask_price_1 columns
data_filtered = data[['open', 'high', 'low', 'close', 'bid_price_1', 'ask_price_1']]

# Calculate the number of missing values for each column in data_filtered
missing_values = data_filtered.isnull().sum()

# Display the missing values for each column
print("Missing values for each column:")
print(missing_values)

# Drop rows with missing values
data_filtered = data_filtered.dropna()
data_filtered

Missing values for each column:
open           157
high           157
low            157
close          157
bid_price_1      0
ask_price_1      0
dtype: int64


,open,high,low,close,bid_price_1,ask_price_1
timestamp,,,,,,
2023-09-12 13:45:00+00:00,179.480,179.52,179.260,179.260,179.52,179.26
2023-09-12 13:46:00+00:00,179.265,179.31,179.060,179.090,179.31,179.07
2023-09-12 13:47:00+00:00,179.100,179.11,178.870,179.010,179.10,178.87
2023-09-12 13:48:00+00:00,179.010,179.10,178.780,178.820,179.10,178.78
2023-09-12 13:49:00+00:00,178.810,178.83,178.510,178.600,178.82,178.51
...,...,...,...,...,...,...
2024-09-20 19:55:00+00:00,231.950,232.05,229.750,229.770,232.03,229.77
2024-09-20 19:56:00+00:00,229.780,229.82,228.920,229.050,229.81,228.94
2024-09-20 19:57:00+00:00,229.060,229.10,228.435,228.435,229.08,228.44


## SMA(Simple Moving Average)

In [48]:
# Function to forecast future values using an iterative SMA method with a rolling window of 390 data points
def forecast_sma(series, window, steps):
    # Create a copy of the original series to append forecasted values
    extended_series = series.copy()
    
    forecast = []
    
    for step in range(steps):
        # Calculate the rolling mean on the extended series, including forecasted values
        if len(extended_series) >= window:
            sma_value = extended_series[-window:].mean()  # Calculate the SMA using the latest window
        else:
            sma_value = extended_series.mean()  # If the series is too short, just use the mean
        
        # Append the forecasted value to the forecast list
        forecast.append(sma_value)
        
        # Add the forecasted value to the extended series for future SMA calculations using pd.concat
        new_index = extended_series.index[-1] + pd.Timedelta(minutes=1)
        new_series = pd.Series([sma_value], index=[new_index])
        extended_series = pd.concat([extended_series, new_series])
    
    # Convert the forecast list into a Pandas Series with proper timestamp index
    forecast_index = pd.date_range(start=series.index[-1], periods=steps+1, freq='T')[1:]
    return pd.Series(forecast, index=forecast_index)

# Forecasting for next 390 minutes using a 390-minute moving average window
window_size = 390 * 3  # Rolling window of 390 data points
forecast_horizon = 390  # Forecast for the next 390 minutes

# Forecast for 'open', 'high', 'low', 'close', 'bid_price_1', 'ask_price_1'
forecast_open = forecast_sma(data_filtered['open'], window_size, forecast_horizon)
forecast_high = forecast_sma(data_filtered['high'], window_size, forecast_horizon)
forecast_low = forecast_sma(data_filtered['low'], window_size, forecast_horizon)
forecast_close = forecast_sma(data_filtered['close'], window_size, forecast_horizon)
forecast_bid_price = forecast_sma(data_filtered['bid_price_1'], window_size, forecast_horizon)
forecast_ask_price = forecast_sma(data_filtered['ask_price_1'], window_size, forecast_horizon)

# Create a DataFrame to hold the forecast results for all columns
forecast_df = pd.DataFrame({
    'open': forecast_open,
    'high': forecast_high,
    'low': forecast_low,
    'close': forecast_close,
    'bid_price_1': forecast_bid_price,
    'ask_price_1': forecast_ask_price
})

# Display the forecasted DataFrame
forecast_df

/var/folders/16/fkny30t96j9cdtccgcqbt79h0000gn/T/ipykernel_83751/1252797166.py:24: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  forecast_index = pd.date_range(start=series.index[-1], periods=steps+1, freq='T')[1:]


,open,high,low,close,bid_price_1,ask_price_1
2024-09-20 20:00:00+00:00,226.676073,226.782453,226.560235,226.680128,226.777342,226.566197
2024-09-20 20:01:00+00:00,226.683839,226.789395,226.567945,226.687086,226.784297,226.573877
2024-09-20 20:02:00+00:00,226.690773,226.796096,226.575037,226.693861,226.790984,226.580957
2024-09-20 20:03:00+00:00,226.697543,226.802434,226.581819,226.700497,226.797336,226.587727
2024-09-20 20:04:00+00:00,226.704131,226.808958,226.588803,226.707523,226.803855,226.594708
...,...,...,...,...,...,...
2024-09-21 02:25:00+00:00,229.082566,229.179955,228.976449,229.085343,229.174854,228.982040
2024-09-21 02:26:00+00:00,229.089885,229.186929,228.983677,229.092493,229.181833,228.989272
2024-09-21 02:27:00+00:00,229.097038,229.194004,228.990894,229.099554,229.188911,228.996494
2024-09-21 02:28:00+00:00,229.104104,229.200964,228.997920,229.106648,229.195884,229.003525


# Holt’s Linear Trend Model

In [49]:
from statsmodels.tsa.holtwinters import Holt
import copy
# Function to forecast future values using Holt's Linear Trend Model
def forecast_holt(series, steps):
    # Fit the Holt's linear trend model
    model = Holt(series, exponential=False)
    model_fit = model.fit()

    # Forecast the future values
    forecast = model_fit.forecast(steps)

    # Ensure the length of forecast and index match
    forecast_index = pd.date_range(start=series.index[-1] + pd.Timedelta(minutes=1), periods=steps, freq='T')

    # Create a dictionary instead of Pandas Series
    forecast_dict = {timestamp: value for timestamp, value in zip(forecast_index, forecast)}

    # Debug print to check the forecasted dictionary
    print(f"Forecast length: {len(forecast)}, Index length: {len(forecast_index)}")

    return forecast_dict

# Forecasting for next 390 minutes
forecast_horizon = 390  # Forecast for the next 390 minutes

# Forecast for 'open', 'high', 'low', 'close', 'bid_price_1', 'ask_price_1'
forecast_open = forecast_holt(data_filtered['open'], forecast_horizon)
forecast_high = forecast_holt(data_filtered['high'], forecast_horizon)
forecast_low = forecast_holt(data_filtered['low'], forecast_horizon)
forecast_close = forecast_holt(data_filtered['close'], forecast_horizon)
forecast_bid_price = forecast_holt(data_filtered['bid_price_1'], forecast_horizon)
forecast_ask_price = forecast_holt(data_filtered['ask_price_1'], forecast_horizon)

# Create a DataFrame to hold the forecast results for all columns
forecast_df = pd.DataFrame({
    'open': forecast_open,
    'high': forecast_high,
    'low': forecast_low,
    'close': forecast_close,
    'bid_price_1': forecast_bid_price,
    'ask_price_1': forecast_ask_price
})

# Display the forecasted DataFrame
forecast_df

/Users/jaden/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Users/jaden/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/base/tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/jaden/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/base/tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/var/folders/16/fkny30t96j9cdtccgcqbt79h0000gn/T/ipykernel_83751/3601729035.py:13: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  forecast_index 

Forecast length: 390, Index length: 390


/Users/jaden/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/base/tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/jaden/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/base/tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/var/folders/16/fkny30t96j9cdtccgcqbt79h0000gn/T/ipykernel_83751/3601729035.py:13: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  forecast_index = pd.date_range(start=series.index[-1] + pd.Timedelta(minutes=1), periods=steps, freq='T')
/Users/jaden/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency

Forecast length: 390, Index length: 390


/Users/jaden/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/base/tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/jaden/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/base/tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/var/folders/16/fkny30t96j9cdtccgcqbt79h0000gn/T/ipykernel_83751/3601729035.py:13: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  forecast_index = pd.date_range(start=series.index[-1] + pd.Timedelta(minutes=1), periods=steps, freq='T')
/Users/jaden/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency

...

Forecast length: 390, Index length: 390


/Users/jaden/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/base/tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/jaden/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/base/tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


Forecast length: 390, Index length: 390


/var/folders/16/fkny30t96j9cdtccgcqbt79h0000gn/T/ipykernel_83751/3601729035.py:13: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  forecast_index = pd.date_range(start=series.index[-1] + pd.Timedelta(minutes=1), periods=steps, freq='T')


,open,high,low,close,bid_price_1,ask_price_1
2024-09-20 20:00:00+00:00,228.313446,228.592049,227.611142,228.519191,228.521696,227.631945
2024-09-20 20:01:00+00:00,228.306669,228.584098,227.602285,228.515675,228.513391,227.623889
2024-09-20 20:02:00+00:00,228.299892,228.576146,227.593427,228.512158,228.505087,227.615834
2024-09-20 20:03:00+00:00,228.293115,228.568195,227.584570,228.508642,228.496782,227.607778
2024-09-20 20:04:00+00:00,228.286338,228.560244,227.575712,228.505125,228.488478,227.599722
...,...,...,...,...,...,...
2024-09-21 02:25:00+00:00,225.704360,225.530834,224.200985,227.165330,225.324511,224.530576
2024-09-21 02:26:00+00:00,225.697583,225.522883,224.192127,227.161813,225.316206,224.522520
2024-09-21 02:27:00+00:00,225.690806,225.514932,224.183270,227.158297,225.307902,224.514465
2024-09-21 02:28:00+00:00,225.684029,225.506981,224.174412,227.154780,225.299598,224.506409
